# Clasificación de Imágenes de Moda usando Descriptores y Clasificador por Centroides

En este proyecto se desarrolla un sistema de clasificación de imágenes utilizando técnicas básicas de visión por computador. El objetivo es identificar tres categorías de prendas:

- Bolsos
- Pantalones
- Zapatos

Para realizar la clasificación se utilizan descriptores de características extraídos de las imágenes, incluyendo información estadística, de forma y de textura. Posteriormente se aplica un clasificador basado en centroides utilizando distancia cuadrática.

El conjunto de datos se divide en dos partes:

- 300 imágenes por clase para entrenamiento
- El resto de las imágenes para evaluación del modelo (Total de imagenes 5998)

Finalmente se evalúa el desempeño del sistema mediante porcentajes de clasificación y matriz de confusión.

# Importar librerias 

In [12]:
import cv2
import numpy as np
import pandas as pd
import os
from sklearn.preprocessing import StandardScaler

## Librerías utilizadas

Para el desarrollo del sistema de clasificación se utilizan las siguientes librerías:

*OpenCV (cv2):*
Se utiliza para el procesamiento de imágenes, incluyendo lectura de archivos, conversión a escala de grises, segmentación y cálculo de contornos.

*NumPy:*
Permite trabajar con matrices y realizar operaciones matemáticas necesarias para calcular los descriptores de las imágenes.

*Pandas:*
Se utiliza para organizar los resultados en forma de tablas, especialmente en la construcción de la matriz de confusión.

*OS:*
Permite acceder a las carpetas del sistema donde se encuentran almacenadas las imágenes del dataset.

*StandardScaler (scikit-learn):*
Se utiliza para normalizar los descriptores de las imágenes antes de realizar la clasificación. Esto evita que variables con valores grandes dominen el cálculo de distancia.


# KERNELS SOBEL VISIBLES

In [13]:


kernel_sobel_x = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1]
], dtype=np.float64)

kernel_sobel_y = np.array([
    [-1, -2, -1],
    [ 0,  0,  0],
    [ 1,  2,  1]
], dtype=np.float64)

## Operador de Sobel

Para detectar bordes en las imágenes se utiliza el operador Sobel.

Este operador calcula el gradiente de intensidad de la imagen en dos direcciones:

- Dirección horizontal (Sobel X)
- Dirección vertical (Sobel Y)

Los bordes de los objetos contienen información importante sobre la forma de las prendas. Al calcular la magnitud del gradiente se obtiene una medida de la presencia de bordes en la imagen.

Esta información posteriormente se utiliza como uno de los descriptores para caracterizar cada imagen.


# FUNCION DESCRIPTOR


In [14]:


def descriptor_individual(imagen):

    imagen = cv2.resize(imagen, (128,128))
    gray = cv2.cvtColor(imagen, cv2.COLOR_BGR2GRAY)

    # -------- ELIMINAR FONDO NEGRO --------
    _, mascara = cv2.threshold(gray, 16, 255, cv2.THRESH_BINARY)
    pixeles = gray[mascara == 255]

    if len(pixeles) == 0:
        pixeles = gray.flatten()

    # -------- DESCRIPTORES --------
    media = np.mean(pixeles)
    varianza = np.var(pixeles)
    desviacion = np.std(pixeles)

    # -------- AREA --------
    area = np.sum(mascara == 255)

    # -------- PERIMETRO --------
    contornos, _ = cv2.findContours(mascara, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    perimetro = 0
    for c in contornos:
        perimetro += cv2.arcLength(c, True)

    # -------- GRADIENTE --------
    sobelx = cv2.filter2D(gray.astype(np.float64), cv2.CV_64F, kernel_sobel_x)
    sobely = cv2.filter2D(gray.astype(np.float64), cv2.CV_64F, kernel_sobel_y)

    magnitud = np.sqrt(sobelx**2 + sobely**2)
    gradiente = np.mean(magnitud)

    # -------- ENTROPIA --------
    hist = cv2.calcHist([gray],[0],None,[256],[0,256])
    hist = hist / np.sum(hist)
    entropia = -np.sum(hist * np.log2(hist + 1e-7))

    descriptor = np.array([
        media,
        varianza,
        desviacion,
        area,
        perimetro,
        gradiente,
        entropia
    ], dtype=np.float64)

    descriptor = np.nan_to_num(descriptor)

    return descriptor



## Descriptores utilizados

Para representar cada imagen se utilizaron diferentes *descriptores de características*. Un descriptor es un valor numérico que resume alguna propiedad importante de la imagen, permitiendo que el algoritmo pueda comparar imágenes entre sí.

Antes de calcular los descriptores se realiza un *preprocesamiento* de la imagen. En este paso se elimina el fondo negro utilizando una máscara binaria generada mediante umbralización. Esto permite que los cálculos se realicen principalmente sobre los píxeles que pertenecen al objeto (la prenda) y no sobre el fondo, evitando que el fondo afecte los valores de los descriptores.

Los descriptores utilizados fueron los siguientes:

- *Media*  
  Representa el valor promedio de intensidad de los píxeles del objeto. Se escogió porque permite tener una idea general del nivel de brillo o tonalidad de la prenda.

- *Varianza*  
  Mide qué tan dispersos están los valores de intensidad respecto a la media. Se eligió porque ayuda a identificar diferencias en la distribución de los tonos dentro de la imagen.

- *Desviación estándar*  
  Es otra medida de dispersión relacionada con la varianza. Se utiliza para complementar la información sobre la variabilidad de intensidades en la imagen.

- *Área*  
  Corresponde al número de píxeles que pertenecen al objeto después de eliminar el fondo. Se escogió porque permite capturar el *tamaño aproximado de la prenda*, lo cual puede ayudar a diferenciar categorías.

- *Perímetro*  
  Representa la longitud del contorno del objeto detectado en la imagen. Se eligió porque describe la *forma del objeto*, lo que ayuda a distinguir prendas con siluetas diferentes.

- *Gradiente (Sobel)*  
  Se calcula utilizando los filtros de Sobel en dirección horizontal y vertical. Este descriptor mide la presencia de *bordes en la imagen*, lo cual es útil para identificar las formas características de cada prenda.

- *Entropía*  
  Mide la complejidad o textura de la imagen a partir de su histograma de intensidades. Se escogió porque permite capturar información sobre la *variabilidad de patrones o detalles en la prenda*.

En conjunto, estos descriptores permiten representar cada imagen mediante un *vector de características*, el cual resume propiedades importantes como intensidad, forma, bordes y textura. Este vector es posteriormente utilizado por el clasificador para comparar imágenes y determinar a qué categoría pertenecen.


# CARGAR SOLO LAS PRIMERAS 300 IMAGENES DEL DATASET


In [15]:


def cargar_categoria(ruta, max_imagenes=300):

    archivos = sorted(os.listdir(ruta))
    datos = []

    for archivo in archivos[:max_imagenes]:

        img = cv2.imread(os.path.join(ruta, archivo))

        if img is not None:
            datos.append(descriptor_individual(img))

    return np.array(datos)


bolso = cargar_categoria("dataset/bolso", 300)
pantalon = cargar_categoria("dataset/pantalon", 300)
zapatos = cargar_categoria("dataset/zapatos", 300)

print("Cantidad Bolso:", len(bolso))
print("Cantidad Pantalon:", len(pantalon))
print("Cantidad Zapatos:", len(zapatos))

Cantidad Bolso: 300
Cantidad Pantalon: 300
Cantidad Zapatos: 300


## Carga del dataset

Las imágenes se encuentran organizadas en tres carpetas:

- dataset/bolso
- dataset/pantalon
- dataset/zapatos

De cada categoría se utilizan las primeras 300 imágenes para construir el conjunto de entrenamiento.

Estas imágenes se utilizan para calcular los centroides de cada clase.

# NORMALIZACION DE LOS DESCRIPTORES

In [16]:
X = np.vstack((bolso, pantalon, zapatos))

scaler = StandardScaler()
X = scaler.fit_transform(X)

print("\nMatriz NORMALIZADA:")
print(X)

n_b = len(bolso)
n_p = len(pantalon)

bolso = X[:n_b]
pantalon = X[n_b:n_b+n_p]
zapatos = X[n_b+n_p:]



Matriz NORMALIZADA:
[[ 0.90466602  0.10249983  0.19239537 ...  0.20048693  0.78376471
   1.59044543]
 [-0.50281387  0.38604971  0.45813099 ...  1.72216595  3.94221708
   1.5873451 ]
 [ 0.282948   -0.78830431 -0.74165787 ... -0.63771806  1.07538958
   0.55584745]
 ...
 [ 0.89344417  0.25685365  0.33859897 ... -1.35823834 -0.40143674
  -0.85160115]
 [ 1.04642586  1.82657041  1.65351924 ... -1.21240134 -0.99578499
  -1.00356948]
 [-0.27912134 -0.14750856 -0.05301125 ... -0.57242462  0.20780239
   0.33109031]]


## Normalización de descriptores

Una vez calculados los descriptores para cada imagen, es necesario realizar un proceso de *normalización*.

Los descriptores obtenidos tienen escalas muy diferentes. Por ejemplo, el área del objeto puede tener valores de miles de píxeles, mientras que la entropía suele tener valores pequeños. Si se calculan distancias directamente con estos valores, los descriptores de mayor magnitud dominarían el cálculo de distancia.

Para evitar este problema se utiliza *StandardScaler*, el cual transforma cada característica para que tenga:

- media igual a 0
- desviación estándar igual a 1

La transformación se realiza mediante la siguiente ecuación:

z = (x − μ) / σ

donde:

- x es el valor original del descriptor
- μ es la media de ese descriptor
- σ es la desviación estándar

Después de la normalización, todos los descriptores quedan en una escala comparable, lo cual mejora el comportamiento del clasificador basado en distancias.

Primero se combinan los descriptores de todas las clases en una sola matriz. Luego se aplica la normalización y finalmente se separan nuevamente los datos correspondientes a cada categoría.


# CENTROIDES Y DISTANCIA CUADRATICA

In [17]:
centro_bolso = np.mean(bolso, axis=0)
centro_pantalon = np.mean(pantalon, axis=0)
centro_zapatos = np.mean(zapatos, axis=0)

print("\nCENTROIDES NORMALIZADOS")

print("\nCentroide Bolso:")
print(centro_bolso)

print("\nCentroide Pantalon:")
print(centro_pantalon)

print("\nCentroide Zapatos:")
print(centro_zapatos)

# DISTANCIA CUADRATICA

def distancia(x, centro):
    return np.sum((x - centro)**2)


CENTROIDES NORMALIZADOS

Centroide Bolso:
[ 0.25833908 -0.39741349 -0.38525226  1.11903836 -0.00157522  0.70933372
  1.17131623]

Centroide Pantalon:
[ 0.21740616  0.02554086  0.04683996 -0.41995984  0.15927347  0.04642263
 -0.61073963]

Centroide Zapatos:
[-0.47574524  0.37187262  0.3384123  -0.69907851 -0.15769825 -0.75575635
 -0.5605766 ]


## Cálculo de centroides

Después de normalizar los descriptores, se calcula un *centroide para cada categoría* (bolso, pantalón y zapatos).

Un *centroide* es el punto promedio que representa a todas las imágenes de una clase dentro del espacio de características.

Para obtenerlo se hace lo siguiente:

- Cada imagen está representada por un *vector de descriptores* (media, varianza, área, perímetro, gradiente, etc.).
- Para cada descriptor se calcula el *promedio de las 300 imágenes de entrenamiento*.
- El resultado es un nuevo vector que representa el *comportamiento promedio de esa categoría*.

De esta forma se obtienen tres centroides:

- Centroide de *bolsos*
- Centroide de *pantalones*
- Centroide de *zapatos*

Cada uno de estos centroides sirve como *referencia* para clasificar nuevas imágenes.


## Clasificación usando distancia cuadrática

Una vez calculados los centroides, se puede clasificar una imagen nueva.

El proceso es el siguiente:

- Primero se calculan los *descriptores de la nueva imagen*.
- Luego se *normalizan* usando el mismo escalador.
- Después se calcula la *distancia entre la imagen y cada centroide*.

La distancia utilizada es la *distancia cuadrática*, que mide qué tan lejos está un vector respecto a otro.

El procedimiento es:

- Se resta cada descriptor de la imagen con el descriptor del centroide.
- Se eleva al cuadrado la diferencia.
- Se suman todos los valores.

El resultado es un número que representa la distancia.

Finalmente:

- La imagen se asigna a la *clase cuyo centroide tenga la menor distancia*.

Por ejemplo:

- Si la distancia menor es con el centroide de *bolso, la imagen se clasifica como **bolso*.
- Si la menor distancia es con *pantalón, se clasifica como **pantalón*.
- Si es menor con *zapatos, se clasifica como **zapatos*.

### Cantidad de valores en los centroides

En el terminal se observan *7 valores para cada centroide. Esto ocurre porque cada imagen es representada mediante un **vector de 7 descriptores*.  

Los descriptores calculados para cada imagen son:

1. Media  
2. Varianza  
3. Desviación estándar  
4. Área  
5. Perímetro  
6. Gradiente (Sobel)  
7. Entropía  

Cada uno de estos valores representa una característica de la imagen. Cuando se calcula el *centroide de una categoría, lo que se hace es obtener el **promedio de cada descriptor* para todas las imágenes de esa clase.

Por esta razón, el centroide final queda formado por *7 valores*, donde cada posición corresponde al promedio de uno de los descriptores utilizados. De esta manera, cada centroide representa el comportamiento promedio de las características de las imágenes de su categoría.

# PROBAR IMAGEN NUEVA

In [18]:

ruta_test = "dataset/bolso/bolso_23.jpg"
imagen_test = cv2.imread(ruta_test)

if imagen_test is None:

    print("Error cargando imagen")

else:

    x_nueva = descriptor_individual(imagen_test)

    print("\nDescriptor de la imagen de prueba:")
    print(x_nueva)

    x_nueva = scaler.transform([x_nueva])

    print("\nDescriptor NORMALIZADO:")
    print(x_nueva)

    d_b = distancia(x_nueva, centro_bolso)
    d_p = distancia(x_nueva, centro_pantalon)
    d_z = distancia(x_nueva, centro_zapatos)

    print("\nDistancias:")
    print("Bolso:", d_b)
    print("Pantalon:", d_p)
    print("Zapatos:", d_z)

    categoria = min(
        {"Bolso": d_b, "Pantalon": d_p, "Zapatos": d_z},
        key=lambda k: {"Bolso": d_b, "Pantalon": d_p, "Zapatos": d_z}[k]
    )

    print("\nClasificacion final:", categoria)


Descriptor de la imagen de prueba:
[1.33854669e+02 1.76756061e+03 4.20423669e+01 1.21860000e+04
 4.55112698e+02 7.23166561e+01 7.06842995e+00]

Descriptor NORMALIZADO:
[[-0.27866561 -1.38419278 -1.4915621   2.26495413  0.08736382  1.83232834
   2.11367532]]

Distancias:
Bolso: 5.95621952096493
Pantalon: 22.42594707251992
Zapatos: 29.16676296050579

Clasificacion final: Bolso


## Prueba del clasificador con una imagen del dataset

Para verificar el funcionamiento del sistema de clasificación, se carga una imagen de prueba tomada del dataset. En este caso se selecciona una imagen de la carpeta correspondiente a una de las categorías.

Una vez cargada la imagen, se calculan sus descriptores utilizando la misma función empleada durante el entrenamiento. Posteriormente, estos descriptores se normalizan utilizando el mismo StandardScaler aplicado previamente, con el fin de mantener la misma escala utilizada para calcular los centroides.

Luego se calculan las distancias entre el vector de características de la imagen y cada uno de los centroides de las categorías:

- centroide de bolsos  
- centroide de pantalones  
- centroide de zapatos  

Estas distancias indican qué tan similar es la imagen respecto a cada clase.

## Clasificación final

Finalmente, el algoritmo compara las tres distancias obtenidas y selecciona la categoría cuya distancia sea la menor. Esto significa que la imagen será clasificada en la clase cuyo centroide sea más cercano a su vector de características.

El resultado se muestra en pantalla indicando la *categoría final asignada a la imagen de prueba*.

# EVALUAR DATASET RESTANTE

In [19]:


def evaluar_categoria(ruta, centro_b, centro_p, centro_z, inicio=300):

    archivos = sorted(os.listdir(ruta))

    total = 0
    correctos = 0

    for archivo in archivos[inicio:]:

        img = cv2.imread(os.path.join(ruta, archivo))

        if img is None:
            continue

        x = descriptor_individual(img)
        x = scaler.transform([x])

        d_b = distancia(x, centro_b)
        d_p = distancia(x, centro_p)
        d_z = distancia(x, centro_z)

        pred = min(
            {"Bolso": d_b, "Pantalon": d_p, "Zapatos": d_z},
            key=lambda k: {"Bolso": d_b, "Pantalon": d_p, "Zapatos": d_z}[k]
        )

        total += 1

        if pred.lower() in ruta.lower():
            correctos += 1

    porcentaje = (correctos / total) * 100

    return porcentaje, total

# CALCULAR PORCENTAJES


acc_bolso, total_b = evaluar_categoria("dataset/bolso", centro_bolso, centro_pantalon, centro_zapatos)

acc_pantalon, total_p = evaluar_categoria("dataset/pantalon", centro_bolso, centro_pantalon, centro_zapatos)

acc_zapatos, total_z = evaluar_categoria("dataset/zapatos", centro_bolso, centro_pantalon, centro_zapatos)




## Evaluación del dataset restante

Después de entrenar el modelo con 300 imágenes por categoría, es necesario evaluar qué tan bien funciona el clasificador. Para esto se utilizan *las imágenes restantes del dataset*, es decir, todas las que no se usaron en el entrenamiento.

La función evaluar_categoria() se encarga de realizar esta evaluación.

El procedimiento es el siguiente:

- Primero se obtienen todos los archivos de imágenes de la carpeta correspondiente a una categoría utilizando os.listdir().
- Luego se inicializan dos variables:
  - total: contador del número total de imágenes evaluadas.
  - correctos: contador de imágenes clasificadas correctamente.

Después el algoritmo recorre las imágenes restantes del dataset (desde la posición 300 en adelante), ya que las primeras 300 se utilizaron para el entrenamiento.

Para cada imagen se realiza el siguiente proceso:

- Se carga la imagen desde la carpeta correspondiente.
- Se calculan sus *descriptores* usando la función descriptor_individual().
- El vector de características se *normaliza* usando el mismo StandardScaler.
- Se calculan las *distancias entre la imagen y cada uno de los centroides*:
  - centroide de bolso
  - centroide de pantalón
  - centroide de zapatos

Posteriormente se selecciona la categoría cuya distancia sea *la menor*, lo que corresponde a la clase predicha por el modelo.

Luego se actualizan los contadores:

- total aumenta en 1 por cada imagen evaluada.
- Si la categoría predicha coincide con la carpeta real de la imagen, se incrementa el contador correctos.

Finalmente se calcula el *porcentaje de clasificación correcta* mediante la fórmula:

porcentaje = (correctos / total) × 100

Este valor representa la *precisión del clasificador para esa categoría*, indicando qué porcentaje de imágenes fue clasificado correctamente.

# TABLA DE RESULTADOS


In [20]:

print("\n==============================")
print("RESULTADOS DE CLASIFICACION")
print("==============================")

print(f"Bolso     -> {acc_bolso:.2f}%  de {total_b} imagenes")
print(f"Pantalon  -> {acc_pantalon:.2f}%  de {total_p} imagenes")
print(f"Zapatos   -> {acc_zapatos:.2f}%  de {total_z} imagenes")

print("\nPromedio general:")

promedio = (acc_bolso + acc_pantalon + acc_zapatos) / 3

print(f"{promedio:.2f}%")




RESULTADOS DE CLASIFICACION
Bolso     -> 85.78%  de 5698 imagenes
Pantalon  -> 81.05%  de 5698 imagenes
Zapatos   -> 80.27%  de 5698 imagenes

Promedio general:
82.37%


## Tabla de resultados de clasificación

Después de evaluar todas las imágenes restantes del dataset, se muestran los resultados obtenidos para cada categoría.

En esta sección del código se imprimen en pantalla los *porcentajes de clasificación correcta* para cada clase.

El procedimiento es el siguiente:

- Primero se imprime un título para organizar la salida en la consola y facilitar la lectura de los resultados.
- Luego se muestran los porcentajes de clasificación para cada categoría:
  - *Bolso*
  - *Pantalón*
  - *Zapatos*

Cada línea muestra dos datos importantes:

- El *porcentaje de aciertos*, que indica qué proporción de imágenes fue clasificada correctamente.
- El *número total de imágenes evaluadas* para esa categoría.

Por ejemplo, un resultado como:

Bolso → 85% de 5699 imágenes

significa que el 85% de las imágenes de bolsos del conjunto de prueba fueron clasificadas correctamente por el sistema.

---

## Promedio general del modelo

Además del desempeño individual por categoría, también se calcula un *promedio general de clasificación*.

Para esto se suman los porcentajes obtenidos en cada categoría y se dividen entre el número total de clases (3).

Este promedio permite tener una *medida global del desempeño del clasificador*, indicando qué tan bien funciona el modelo en general para todas las categorías del dataset.

# MATRIZ DE CONFUSION

In [21]:


def matriz_confusion(ruta, etiqueta_real, inicio=300):

    archivos = sorted(os.listdir(ruta))

    conteo = {"Bolso":0, "Pantalon":0, "Zapatos":0}

    for archivo in archivos[inicio:]:

        img = cv2.imread(os.path.join(ruta, archivo))

        if img is None:
            continue

        x = descriptor_individual(img)
        x = scaler.transform([x])

        d_b = distancia(x, centro_bolso)
        d_p = distancia(x, centro_pantalon)
        d_z = distancia(x, centro_zapatos)

        pred = min(
            {"Bolso": d_b, "Pantalon": d_p, "Zapatos": d_z},
            key=lambda k: {"Bolso": d_b, "Pantalon": d_p, "Zapatos": d_z}[k]
        )

        conteo[pred] += 1

    return conteo

## Cálculo de la matriz de confusión

Para analizar con mayor detalle el comportamiento del clasificador se construye una *matriz de confusión. Esta matriz permite observar no solo cuántas imágenes fueron clasificadas correctamente, sino también **en qué casos el modelo se equivoca y con qué categorías se confunde*.

La función matriz_confusion() se encarga de contar las predicciones realizadas por el modelo para cada categoría.

El procedimiento es el siguiente:

- Primero se obtienen todas las imágenes de la carpeta correspondiente a una categoría utilizando os.listdir().
- Luego se crea un diccionario llamado conteo que almacenará cuántas imágenes fueron clasificadas como:
  - Bolso
  - Pantalón
  - Zapatos

Después el algoritmo recorre todas las imágenes restantes del dataset (desde la posición 300 en adelante), ya que las primeras 300 se utilizaron para el entrenamiento.

Para cada imagen se realiza el mismo proceso de clasificación:

- Se carga la imagen.
- Se calculan sus *descriptores*.
- Se *normaliza* el vector de características.
- Se calculan las *distancias a los centroides* de cada categoría.
- Se determina la *categoría predicha*, es decir, la clase cuyo centroide tenga la menor distancia.

Una vez obtenida la predicción, el algoritmo incrementa el contador correspondiente dentro del diccionario conteo. De esta forma se registra cuántas imágenes fueron clasificadas como cada categoría.

Finalmente, la función devuelve este conteo, el cual se utilizará posteriormente para construir la *matriz de confusión completa*, donde se comparan las categorías reales con las categorías predichas por el modelo.

# CALCULAR MATRIZ & MATRIZ EN PORCENTAJES


In [22]:
# CALCULAR MATRIZ


bolso_res = matriz_confusion("dataset/bolso", "Bolso")
pantalon_res = matriz_confusion("dataset/pantalon", "Pantalon")
zapatos_res = matriz_confusion("dataset/zapatos", "Zapatos")

matriz = np.array([
    [bolso_res["Bolso"], bolso_res["Pantalon"], bolso_res["Zapatos"]],
    [pantalon_res["Bolso"], pantalon_res["Pantalon"], pantalon_res["Zapatos"]],
    [zapatos_res["Bolso"], zapatos_res["Pantalon"], zapatos_res["Zapatos"]]
])

categorias = ["Bolso", "Pantalon", "Zapatos"]

df_matriz = pd.DataFrame(matriz, index=categorias, columns=categorias)

print("\n==============================")
print("MATRIZ DE CONFUSION (VALORES)")
print("==============================\n")

print(df_matriz)

# MATRIZ EN PORCENTAJES

porcentajes = matriz / matriz.sum(axis=1, keepdims=True) * 100

df_por = pd.DataFrame(porcentajes, index=categorias, columns=categorias)

print("\n==============================")
print("MATRIZ DE CONFUSION (%)")
print("==============================\n")

print(df_por.round(2))


MATRIZ DE CONFUSION (VALORES)

          Bolso  Pantalon  Zapatos
Bolso      4888       469      341
Pantalon    351      4618      729
Zapatos     125       999     4574

MATRIZ DE CONFUSION (%)

          Bolso  Pantalon  Zapatos
Bolso     85.78      8.23     5.98
Pantalon   6.16     81.05    12.79
Zapatos    2.19     17.53    80.27


## Construcción de la matriz de confusión

Después de evaluar cada categoría por separado, se construye la *matriz de confusión completa* para analizar el comportamiento del clasificador.

Primero se ejecuta la función matriz_confusion() para cada categoría del dataset:

- bolso_res almacena los resultados de las imágenes reales de bolsos.
- pantalon_res almacena los resultados de las imágenes reales de pantalones.
- zapatos_res almacena los resultados de las imágenes reales de zapatos.

Cada uno de estos resultados contiene cuántas imágenes fueron clasificadas como:

- Bolso  
- Pantalón  
- Zapatos  

Posteriormente se organiza esta información en una matriz utilizando numpy. En esta matriz:

- *Las filas representan la categoría real de la imagen.*
- *Las columnas representan la categoría predicha por el modelo.*

La matriz queda organizada de la siguiente forma:

- Primera fila → resultados de imágenes reales de *bolso*  
- Segunda fila → resultados de imágenes reales de *pantalón*  
- Tercera fila → resultados de imágenes reales de *zapatos*

Después, esta matriz se convierte en un *DataFrame de Pandas* para mostrarla de manera más clara y ordenada en forma de tabla.

## Matriz de confusión en valores

La primera tabla que se muestra corresponde a la *matriz de confusión en valores absolutos*.  

Cada número indica cuántas imágenes fueron clasificadas en cada categoría.

En esta matriz:

- Los valores de la *diagonal principal* representan clasificaciones correctas.
- Los valores fuera de la diagonal representan *errores del modelo*.

Por ejemplo, si una imagen real de pantalón aparece en la columna de bolsos, significa que el modelo confundió esa imagen.


## Matriz de confusión en porcentajes

Para facilitar la interpretación de los resultados, también se calcula una *matriz de confusión en porcentajes*.

Para esto se divide cada fila de la matriz entre el número total de imágenes de esa categoría y luego se multiplica por 100.

De esta manera se obtiene el *porcentaje de clasificación para cada clase*, lo que permite analizar más fácilmente el desempeño del modelo.

Por ejemplo:

- Un valor alto en la diagonal indica que la mayoría de las imágenes fueron clasificadas correctamente.
- Valores altos fuera de la diagonal indican que el modelo tiende a confundir esas categorías.

## Conclusión

En esta actividad se implementó un sistema de clasificación de imágenes para tres categorías de prendas: *bolsos, pantalones y zapatos. Para lograrlo, cada imagen fue representada mediante un conjunto de **descriptores de características* que permiten resumir propiedades importantes como intensidad, forma, bordes y textura.

Antes de calcular los descriptores se realizó un *preprocesamiento de la imagen*, eliminando el fondo negro mediante una máscara binaria. Esto permitió que los cálculos se enfocaran principalmente en los píxeles correspondientes al objeto, evitando que el fondo afectara las características extraídas.

Posteriormente, los descriptores fueron *normalizados* para que todas las características estuvieran en la misma escala, lo cual mejora la comparación entre imágenes. A partir de estos vectores de características se calcularon los *centroides de cada categoría*, que representan el comportamiento promedio de las imágenes de cada clase.

La clasificación se realizó utilizando la *distancia cuadrática*, comparando el descriptor de cada imagen con los centroides de las categorías y asignando la clase correspondiente al centroide más cercano.

Finalmente, el rendimiento del sistema fue evaluado mediante una *matriz de confusión*, obteniendo los siguientes porcentajes de clasificación correcta:

- *Bolso:* 85.78 %  
- *Pantalón:* 81.05 %  
- *Zapatos:* 80.27 %

Estos resultados indican que el sistema logra identificar correctamente la mayoría de las imágenes, especialmente en la categoría de bolsos. Aunque el desempeño en pantalones y zapatos es ligeramente menor, el modelo demuestra que los descriptores seleccionados permiten diferenciar las categorías de manera adecuada.

En general, el método basado en *descriptores simples y distancia a centroides* muestra ser una estrategia efectiva para un problema básico de clasificación de imágenes. Sin embargo, los resultados podrían mejorarse incorporando más descriptores, técnicas de segmentación más avanzadas o modelos de aprendizaje más complejos.